In [2]:
!pip install flwr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 727.1/727.1 kB 17.3 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 83.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 16.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.3/323.3 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.75.1
    Uninstalling grpcio-1.75.1:
      Successfully uninstalled grpcio-1.75.1
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:


In [5]:
!pip uninstall -y protobuf
!pip install protobuf==6.33.4

Found existing installation: protobuf 6.33.4
Uninstalling protobuf-6.33.4:
  Successfully uninstalled protobuf-6.33.4
  Using cached protobuf-6.33.4-cp39-abi3-manylinux2014_x86_64.whl.metadata (593 bytes)
Using cached protobuf-6.33.4-cp39-abi3-manylinux2014_x86_64.whl (323 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 6.33.4 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.4 which is incompatible.
tensorflow 2.19.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!

In [3]:
import numpy as np
import google.protobuf
import torch
import flwr

print("NumPy:", np.__version__)
print("Protobuf:", google.protobuf.__version__)
print("CUDA:", torch.cuda.is_available())
print("Flower:", flwr.__version__)

NumPy: 2.0.2
Protobuf: 5.29.5
CUDA: True
Flower: 1.25.0


In [18]:
# Federated Learning framework
import flwr as fl
from flwr.server.strategy import FedAvg
from flwr.common import Context

# Core PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F

# Data handling
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset

# Utilities
import numpy as np
import csv
import os

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: Tesla P100-PCIE-16GB


In [6]:
# =========================
# Federated configuration
# =========================

NUM_CLIENTS = 3        # Number of federated clients (farmers)
NUM_ROUNDS = 5         # Number of federated rounds (can increase to 10 later)
LOCAL_EPOCHS = 1       # Local epochs per client per round
BATCH_SIZE = 32        # Same as Week 3

# =========================
# Device configuration
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        
        # Convolutional layers
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(64 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [8]:
# =========================
# Model initialization
# =========================

NUM_CLASSES = 38  # Same as Week 3 (PlantVillage)

# Create global model
global_model = SimpleCNN(NUM_CLASSES).to(device)

# Load Week 3 trained weights
MODEL_PATH = "/kaggle/input/simplecnn-weights-plantvillage/week3_simplecnn.pth"
global_model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

# Set to evaluation mode by default (training happens inside clients)
global_model.eval()

print("Global model initialized with Week 3 weights.")


Global model initialized with Week 3 weights.


In [9]:
# =========================
# Dataset & Transforms
# =========================

DATA_PATH = "/kaggle/input/plantvillage-dataset/color"

# Training transforms (same as Week 3)
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
])

# Validation/Test transforms (same as Week 3)
test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load full dataset (no split yet)
full_dataset = datasets.ImageFolder(root=DATA_PATH)

class_names = full_dataset.classes
num_classes = len(class_names)

print("Number of classes:", num_classes)
print("Total dataset size:", len(full_dataset))

Number of classes: 38
Total dataset size: 54305


In [10]:
# =========================
# Federated data splitting (FIXED)
# =========================

indices = np.arange(len(full_dataset))
splits = np.array_split(indices, NUM_CLIENTS)

client_train_datasets = []
client_test_datasets = []

for i, client_indices in enumerate(splits):

    # IMPORTANT: separate datasets so transforms do not overwrite each other
    train_dataset = ImageFolder(
        root=DATA_PATH,
        transform=train_transforms
    )

    test_dataset = ImageFolder(
        root=DATA_PATH,
        transform=test_transforms
    )

    split = int(0.8 * len(client_indices))
    train_idx = client_indices[:split]
    test_idx = client_indices[split:]

    client_train_datasets.append(Subset(train_dataset, train_idx))
    client_test_datasets.append(Subset(test_dataset, test_idx))

    print(
        f"Client {i} -> Train: {len(train_idx)} | Test: {len(test_idx)}"
    )

Client 0 -> Train: 14481 | Test: 3621
Client 1 -> Train: 14481 | Test: 3621
Client 2 -> Train: 14480 | Test: 3621


In [11]:
# =========================
# Client DataLoaders
# =========================

client_train_loaders = []
client_test_loaders = []

for i in range(NUM_CLIENTS):
    train_loader = DataLoader(
        client_train_datasets[i],
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0
    )

    test_loader = DataLoader(
        client_test_datasets[i],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    client_train_loaders.append(train_loader)
    client_test_loaders.append(test_loader)

    print(
        f"Client {i} loaders -> "
        f"Train batches: {len(train_loader)}, "
        f"Test batches: {len(test_loader)}"
    )

Client 0 loaders -> Train batches: 453, Test batches: 114
Client 1 loaders -> Train batches: 453, Test batches: 114
Client 2 loaders -> Train batches: 453, Test batches: 114


In [12]:
# =========================
# Local training function
# =========================

def train_one_epoch(model, train_loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    train_acc = 100 * correct / total
    avg_loss = running_loss / len(train_loader)

    return avg_loss, train_acc


# =========================
# Local evaluation function
# =========================

def evaluate_model(model, test_loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    test_acc = 100 * correct / total
    avg_loss = running_loss / len(test_loader)

    return avg_loss, test_acc

In [13]:
def get_model_parameters(model):
    return [val.detach().cpu().numpy() for _, val in model.state_dict().items()]

def set_model_parameters(model, parameters):
    params_dict = zip(model.state_dict().keys(), parameters)
    state_dict = {
        k: torch.tensor(v, device=device)
        for k, v in params_dict
    }
    model.load_state_dict(state_dict, strict=True)

In [14]:
# =========================
# Flower Client Definition
# =========================

class FlowerClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, test_loader):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.criterion = nn.CrossEntropyLoss()
    
    def get_parameters(self, config=None):
        return get_model_parameters(self.model)

    def fit(self, parameters, config=None):
        set_model_parameters(self.model, parameters)
    
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=1e-4
        )
        
        train_one_epoch(
            self.model,
            self.train_loader,
            optimizer,
            self.criterion,
        )
        
        return (
            get_model_parameters(self.model),
            len(self.train_loader.dataset),
            {},
        )



    def evaluate(self, parameters, config=None):
        set_model_parameters(self.model, parameters)

        loss, acc = evaluate_model(
            self.model,
            self.test_loader,
            self.criterion,
        )

        return loss, len(self.test_loader.dataset), {"accuracy": acc}


In [15]:
# =========================
# Client generator
# =========================
def client_fn(cid: str):
    cid = int(cid)

    # Create model
    model = SimpleCNN(NUM_CLASSES).to(device)

    # Initialize with global weights
    model.load_state_dict(global_model.state_dict())

    # Freeze feature extractor
    for param in model.conv1.parameters():
        param.requires_grad = False
    for param in model.conv2.parameters():
        param.requires_grad = False

    # Create ONE client instance
    client = FlowerClient(
        model=model,
        train_loader=client_train_loaders[cid],
        test_loader=client_test_loaders[cid],
    )

    return client.to_client()

In [16]:
def weighted_average(metrics):
    # metrics: List[(num_examples, Dict[str, float])]
    total_examples = sum(num_examples for num_examples, _ in metrics)

    return {
        "accuracy": sum(
            num_examples * m["accuracy"]
            for num_examples, m in metrics
        ) / total_examples
    }

In [17]:
# =========================
# Federated training
# =========================

strategy = FedAvg(
    fraction_fit=1.0,       # All clients participate in training
    fraction_evaluate=1.0,  # All clients participate in evaluation
    min_fit_clients=NUM_CLIENTS,
    min_evaluate_clients=NUM_CLIENTS,
    min_available_clients=NUM_CLIENTS,
    evaluate_metrics_aggregation_fn=weighted_average,
)

# Start federated learning simulation
history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=NUM_CLIENTS,
    config=fl.server.ServerConfig(num_rounds=NUM_ROUNDS),
    strategy=strategy,
    client_resources={
        "num_cpus": 1,
        "num_gpus": 1.0,  # 👈 REQUIRED
    },
)

2026-01-28 04:12:27.177303: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769573547.377334      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769573547.431696      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769573547.893179      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769573547.893213      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769573547.893216      55 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
INFO :      Starting Flower simulation, config: num_rounds=5, no round_timeout
2026-01-28 04:12:45,938	INFO worker.py:2023 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is depreca

In [19]:
# =========================
# Final results & comparison
# =========================
WEEK3_ACC = 85.49
# Extract final round client metrics
final_round, mean_acc = history.metrics_distributed["accuracy"][-1]

print("========== Week 4 Federated Results ==========")
print(f"Final Federated Accuracy: {mean_acc:.2f}%")

print("---------------------------------------------")
print(f"Week 3 Centralized Accuracy: {WEEK3_ACC:.2f}%")
print(f"Week 4 Federated Accuracy:   {mean_acc:.2f}%")

if mean_acc >= WEEK3_ACC - 5:
    print("✅ System Robust. Deployment Ready.")
else:
    print("❌ Drastic Drop Detected. Increase Rounds or Epochs.")

========== Week 4 Federated Results ==========
Final Federated Accuracy: 87.33%
---------------------------------------------
Week 3 Centralized Accuracy: 85.49%
Week 4 Federated Accuracy:   87.33%
✅ System Robust. Deployment Ready.


In [20]:
# =========================
# Save final global model
# =========================

MODEL_DIR = "models"
os.makedirs(MODEL_DIR, exist_ok=True)

FINAL_MODEL_PATH = os.path.join(MODEL_DIR, "federated_global_model.pth")

torch.save(global_model.state_dict(), FINAL_MODEL_PATH)

print(f"✅ Global federated model saved to {FINAL_MODEL_PATH}")

✅ Global federated model saved to models/federated_global_model.pth


In [21]:
# =========================
# Save training metrics
# =========================

METRICS_DIR = "metrics"
os.makedirs(METRICS_DIR, exist_ok=True)

METRICS_PATH = os.path.join(METRICS_DIR, "federated_metrics.csv")

accuracy_history = history.metrics_distributed["accuracy"]

with open(METRICS_PATH, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["round", "global_accuracy"])

    for round_num, acc in accuracy_history:
        writer.writerow([round_num, acc])

print(f"✅ Training metrics saved to {METRICS_PATH}")

✅ Training metrics saved to metrics/federated_metrics.csv
